# 01 · Base de avaliações e qualidade dos dados

Este notebook abre o projeto da Fase 4. O objetivo é conhecer a base sobre a qual o RAG vai ser construído: quais tabelas da Olist entram, como elas se ligam, qual é o grão da base de conhecimento, onde estão os valores ausentes e o que a nota e a entrega dizem sobre o texto que vai ser indexado.

A base é gerada por `make base` (`src/data/base.py`) e fica em `data/processed/avaliacoes.parquet`. As regras de junção são testadas em `tests/test_base.py` e as colunas estão descritas em `docs/dicionario_de_dados.md`.

In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import numpy as np
import pandas as pd
from scipy import stats

from src.data.download import check_against_manifest, load_manifest
from src.data.olist import TABLES, read_all

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
BASE = Path("data/processed/avaliacoes.parquet")

## 1. As tabelas da Olist

A base pública tem nove CSVs. Seis entram no projeto; pagamentos, vendedores e geolocalização ficam fora porque nenhuma das perguntas de negócio do desafio depende deles (motivos em `docs/fontes_de_dados.md`). A cópia local é conferida contra o SHA-256 gravado no manifesto.

In [2]:
tables = read_all()
uso = {
    "order_reviews": "base do RAG", "orders": "metadados de entrega", "order_items": "valor e categoria",
    "products": "categoria", "category_translation": "conferência", "customers": "UF",
    "order_payments": "fora", "sellers": "fora", "geolocation": "fora",
}
resumo = pd.DataFrame({
    "arquivo": {n: TABLES[n]["file"] for n in tables},
    "linhas": {n: len(df) for n, df in tables.items()},
    "colunas": {n: df.shape[1] for n, df in tables.items()},
    "uso": uso,
})
resumo

,arquivo,linhas,colunas,uso
customers,olist_customers_dataset.csv,99441,5,UF
geolocation,olist_geolocation_dataset.csv,1000163,5,fora
order_items,olist_order_items_dataset.csv,112650,7,valor e categoria
order_payments,olist_order_payments_dataset.csv,103886,5,fora
order_reviews,olist_order_reviews_dataset.csv,99224,7,base do RAG
orders,olist_orders_dataset.csv,99441,8,metadados de entrega
products,olist_products_dataset.csv,32951,9,categoria
sellers,olist_sellers_dataset.csv,3095,4,fora
category_translation,product_category_name_translation.csv,71,2,conferência


In [3]:
manifest = load_manifest()
print("versão do Kaggle:", manifest["versao_kaggle"], "| arquivos no manifesto:", len(manifest["files"]))
print("divergências contra o manifesto:", check_against_manifest() or "nenhuma")

versão do Kaggle: 2 | arquivos no manifesto: 9


divergências contra o manifesto: nenhuma


## 2. Do pedido à avaliação: as junções

O desafio pede respostas fundamentadas em avaliações, então o grão da base de conhecimento é a avaliação. A tabela de avaliações, porém, não tem uma linha por avaliação, e os pedidos não têm uma avaliação cada. A tabela abaixo mede os desvios antes de qualquer regra.

In [4]:
reviews, orders, items = tables["order_reviews"], tables["orders"], tables["order_items"]

por_review = reviews.groupby("review_id")["order_id"].nunique()
por_pedido = reviews.groupby("order_id")["review_id"].nunique()
multi = reviews[reviews["review_id"].isin(por_review[por_review > 1].index)]
cols = ["review_score", "review_comment_title", "review_comment_message", "review_creation_date"]
identicas = multi.groupby("review_id")[cols].nunique(dropna=False).max(axis=1).eq(1)
notas_pedido = reviews[reviews["order_id"].isin(por_pedido[por_pedido > 1].index)]

juncoes = pd.Series({
    "linhas na tabela de avaliações": len(reviews),
    "review_id distintos": reviews["review_id"].nunique(),
    "review_id ligados a mais de um pedido": int((por_review > 1).sum()),
    "  ...com conteúdo idêntico entre os pedidos": int(identicas.sum()),
    "pedidos com mais de uma avaliação": int((por_pedido > 1).sum()),
    "  ...com notas diferentes entre as avaliações": int(notas_pedido.groupby("order_id")["review_score"].nunique().gt(1).sum()),
    "avaliações de pedidos sem itens": int((~reviews["order_id"].isin(items["order_id"])).sum()),
    "pedidos sem avaliação": int((~orders["order_id"].isin(reviews["order_id"])).sum()),
})
juncoes.to_frame("quantidade")

,quantidade
linhas na tabela de avaliações,99224
review_id distintos,98410
review_id ligados a mais de um pedido,789
...com conteúdo idêntico entre os pedidos,789
pedidos com mais de uma avaliação,547
...com notas diferentes entre as avaliações,202
avaliações de pedidos sem itens,759
pedidos sem avaliação,768


- **99.224 linhas viram 98.410 avaliações.** Os 789 `review_id` ligados a mais de um pedido têm nota, título, comentário e datas idênticos em todos os pedidos, e o cliente é sempre o mesmo: é uma resposta única que a Olist associou a várias compras. Indexar uma cópia por pedido faria a mesma frase aparecer repetida entre as evidências de uma resposta, então fica uma linha por avaliação, com os pedidos listados em `order_ids`.
- **Os 547 pedidos com mais de uma avaliação são o caso contrário:** respostas distintas, e em 202 deles com notas diferentes. Todas ficam, porque cada uma é uma fala do cliente; a coluna `order_has_multiple_reviews` permite tratá-las à parte em contagens por pedido.
- **759 linhas (701 avaliações depois da deduplicação) são de pedidos sem itens na base.** Ficam sem categoria e sem valor, mas continuam úteis para perguntas sobre entrega e atendimento.
- Os 768 pedidos sem avaliação não entram: não há o que recuperar.

## 3. A base de avaliações

In [5]:
base = pd.read_parquet(BASE)
print(f"{len(base):,} avaliações".replace(",", "."), f"e {base.shape[1]} colunas")
print("compras de", base["order_purchase_timestamp"].min().date(), "a", base["order_purchase_timestamp"].max().date())
fora = ~base["purchase_month"].between("2017-01", "2018-08")
print(f"fora de jan/2017 a ago/2018: {fora.sum()} avaliações ({100 * fora.mean():.2f}%)")
base.groupby("purchase_year").agg(
    avaliacoes=("review_id", "size"),
    com_comentario=("has_comment", "sum"),
    nota_media=("review_score", "mean"),
).assign(pct_comentario=lambda d: (100 * d["com_comentario"] / d["avaliacoes"]).round(1)).round(2)

98.410 avaliações e 37 colunas
compras de 2016-09-04 a 2018-10-17
fora de jan/2017 a ago/2018: 328 avaliações (0.33%)


,avaliacoes,com_comentario,nota_media,pct_comentario
purchase_year,,,,
2016,323,236,3.53,73.1
2017,44573,18690,4.09,41.9
2018,53514,21715,4.09,40.6


As compras vão de setembro de 2016 a outubro de 2018, mas fora de janeiro de 2017 a agosto de 2018 há só 328 avaliações (0,33%). 2016 tem 323 avaliações e 73% delas com comentário, contra 41% nos anos seguintes: o começo da operação tem poucos pedidos e não serve para comparar períodos. A nota média é a mesma em 2017 e 2018 (4,09).

A distribuição das notas tem o formato típico de avaliações de e-commerce:

In [6]:
notas = base["review_score"].value_counts().sort_index()
pd.DataFrame({"avaliações": notas, "%": (100 * notas / notas.sum()).round(1)})

,avaliações,%
review_score,,
1,11282,11.5
2,3114,3.2
3,8097,8.2
4,19007,19.3
5,56910,57.8


A nota 5 domina (58%), a nota 1 vem em seguida (11,5%) e as notas 2 e 3 são raras. Quem fica insatisfeito tende a ir ao extremo, o que torna "nota 1 ou 2" um recorte natural de insatisfação para o gestor.

## 4. Qualidade: ausentes e inconsistências

In [7]:
chave = ["review_comment_title", "review_comment_message", "order_approved_at",
         "order_delivered_carrier_date", "order_delivered_customer_date", "main_category",
         "customer_state", "items_value"]
ausentes = pd.DataFrame({
    "ausentes": base[chave].isna().sum(),
    "%": (100 * base[chave].isna().mean()).round(1),
})
ausentes

,ausentes,%
review_comment_title,86891,88.3
review_comment_message,57742,58.7
order_approved_at,106,0.1
order_delivered_carrier_date,1679,1.7
order_delivered_customer_date,2848,2.9
main_category,2069,2.1
customer_state,0,0.0
items_value,701,0.7


In [8]:
texto = pd.Series({
    "só nota (sem título nem comentário)": int((~base["has_comment"] & ~base["has_title"]).sum()),
    "só título": int((base["has_title"] & ~base["has_comment"]).sum()),
    "só comentário": int((base["has_comment"] & ~base["has_title"]).sum()),
    "título e comentário": int((base["has_comment"] & base["has_title"]).sum()),
    "comentário preenchido só com espaços": int((base["review_comment_message"].notna() & ~base["has_comment"]).sum()),
})
pd.DataFrame({"avaliações": texto, "%": (100 * texto / len(base)).round(1)})

,avaliações,%
só nota (sem título nem comentário),56040,56.9
só título,1729,1.8
só comentário,30853,31.4
título e comentário,9788,9.9
comentário preenchido só com espaços,27,0.0


In [9]:
sem_cat = base["main_category"].isna()
pd.Series({
    "sem categoria": int(sem_cat.sum()),
    "  ...pedido sem itens na base": int((sem_cat & base["n_items"].eq(0)).sum()),
    "  ...itens de produtos sem categoria": int((sem_cat & base["n_items"].gt(0)).sum()),
    "avaliações com mais de uma categoria": int(base["n_categories"].gt(1).sum()),
}).to_frame("avaliações")

,avaliações
sem categoria,2069
...pedido sem itens na base,701
...itens de produtos sem categoria,1368
avaliações com mais de uma categoria,963


In [10]:
entrega = base["delivery_status"].value_counts()
print(pd.DataFrame({"avaliações": entrega, "%": (100 * entrega / len(base)).round(1)}))
print()
entregues = base[base["delivered"]]
inconsistencias = pd.Series({
    "entrega antes da compra": int((entregues["order_delivered_customer_date"] < entregues["order_purchase_timestamp"]).sum()),
    "entrega ao cliente antes do envio à transportadora": int((entregues["order_delivered_customer_date"] < entregues["order_delivered_carrier_date"]).sum()),
    "status delivered sem data de entrega": int(((base["order_status"] == "delivered") & ~base["delivered"]).sum()),
    "data de entrega com status diferente de delivered": int(((base["order_status"] != "delivered") & base["delivered"]).sum()),
    "respondeu antes de receber (pedido entregue)": int((base["delivered"] & base["answered_before_delivery"]).sum()),
})
inconsistencias.to_frame("avaliações")

                 avaliações     %
delivery_status                  
no prazo              89193  90.6
atrasada               6369   6.5
nao entregue           1661   1.7
cancelada              1187   1.2



,avaliações
entrega antes da compra,0
entrega ao cliente antes do envio à transportadora,23
status delivered sem data de entrega,8
data de entrega com status diferente de delivered,6
respondeu antes de receber (pedido entregue),4770


In [11]:
base.loc[base["delivered"], ["delivery_days", "estimated_days", "delay_days", "answer_days"]].describe(
    percentiles=[0.05, 0.5, 0.95]).round(1)

,delivery_days,estimated_days,delay_days,answer_days
count,95562.0,95562.0,95562.0,95562.0
mean,12.5,23.7,-11.9,3.1
std,9.5,8.8,10.1,9.8
min,0.5,2.0,-147.0,0.1
5%,3.0,10.5,-26.0,0.7
50%,10.2,23.2,-12.0,1.7
95%,29.2,38.4,3.0,7.0
max,208.4,155.1,188.0,518.7


**O que a checagem de qualidade mostra:**

- **Texto.** 57% das avaliações são só a nota, sem título nem comentário. 40.641 avaliações (41,3%) têm comentário, e 1.729 têm só título. Se os títulos sem comentário entram no índice é decidido no notebook 02, junto com a limpeza do texto. Só 27 comentários estão preenchidos com espaços.
- **Ausentes não são imputados.** Título e comentário ausentes são escolha do cliente, não falha de registro: não existe texto a inventar. A data de entrega ausente (2,9%) corresponde a pedido não entregue ou cancelado e vira categoria em `delivery_status`. A categoria ausente (2,1%) vem de 701 avaliações de pedidos sem itens e 1.368 de produtos sem categoria na base; nos filtros ela aparece como "Sem categoria".
- **Inconsistências residuais.** 23 pedidos têm entrega ao cliente registrada antes do envio à transportadora, 8 têm status `delivered` sem data de entrega e 6 têm data de entrega com outro status. Somam 37 de 98.410 e não justificam regra própria: a situação da entrega usa a data de entrega ao cliente.
- **Prazos.** A mediana da entrega é de 10 dias, e a da data estimada informada na compra é de 23 dias. A Olist estima com folga: a entrega mediana chega 12 dias antes da estimativa, e só no percentil 95 o pedido passa 3 dias da data estimada. Por isso um atraso, quando acontece, contraria uma expectativa que o cliente tinha como garantida.
- **Resposta.** O cliente responde a pesquisa em mediana 1,7 dia depois do envio. 4.770 clientes de pedidos que acabaram entregues responderam antes de receber; a hipótese 3 olha esse grupo.

As hipóteses sobre quem comenta, nota e atraso vêm na próxima etapa deste notebook.